In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:\\Users\\janud\\Downloads\\Supply_chain .xlsx")

print(df.head())
print(df.shape)

  Product type   SKU      Price  Availability  Number of products sold  \
0     haircare  SKU0  69.808006            55                      802   
1     skincare  SKU1  14.843523            95                      736   
2     haircare  SKU2  11.319683            34                        8   
3     skincare  SKU3  61.163343            68                       83   
4     skincare  SKU4   4.805496            26                      871   

   Revenue generated Customer demographics  Stock levels  Lead times  \
0        8661.996792            Non-binary            58           7   
1        7460.900065                Female            53          30   
2        9577.749626               Unknown             1          10   
3        7766.836426            Non-binary            23          13   
4        2686.505152            Non-binary             5           3   

   Order quantities  ...  Location Lead time  Production volumes  \
0                96  ...    Mumbai        29          

In [3]:
#Inventory Turnover Proxy
df["Inventory Turnover Proxy"] = (
    df["Number of products sold"]
    / df["Stock levels"]
)

In [5]:
#Stockout Risk Score

df["Stockout Risk Score"] = np.where(
    df["Stock levels"] == 0,
    np.nan,
    (
        df["Number of products sold"]
        / df["Stock levels"]
    ) * df["Lead times"]
)

In [7]:
#Excess Inventory Flag"

stock_median = df["Stock levels"].median()
sales_median = df["Number of products sold"].median()

print("Stock Median:", stock_median)
print("Sales Median:", sales_median)
df["Excess Inventory Flag"] = np.where(
    (df["Stock levels"] > stock_median) &
    (df["Number of products sold"] < sales_median),
    "Yes",
    "No"
)

Stock Median: 47.5
Sales Median: 392.5


In [9]:
print(df["Excess Inventory Flag"].value_counts())

Excess Inventory Flag
No     78
Yes    22
Name: count, dtype: int64


In [11]:
# operational cost 
df["Total Operational Cost"] = (
    df["Manufacturing costs"]
    + df["Shipping costs"]
    +df["Costs"]
)

In [17]:
#Cost Per Unit Sold
df["Cost Per Unit Sold"] = (
    df["Total Operational Cost"]
    / df["Number of products sold"]
)

In [13]:
# Supplier Quality Score
df["Supplier Quality Score"] = (
    100 - df["Defect rates"]
)

In [15]:
print(df.columns)

Index(['Product type', 'SKU', 'Price', 'Availability',
       'Number of products sold', 'Revenue generated', 'Customer demographics',
       'Stock levels', 'Lead times', 'Order quantities', 'Shipping times',
       'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location',
       'Lead time', 'Production volumes', 'Manufacturing lead time',
       'Manufacturing costs', 'Inspection results', 'Defect rates',
       'Transportation modes', 'Routes', 'Costs', 'Inventory Turnover Proxy',
       'Stockout Risk Score', 'Excess Inventory Flag',
       'Total Operational Cost', 'Supplier Quality Score'],
      dtype='object')


In [17]:
print(df["Stockout Risk Score"].describe())

count       99.000000
mean       458.315870
std       1624.759451
min          1.488889
25%         48.580822
50%        132.292683
75%        323.149184
max      15652.000000
Name: Stockout Risk Score, dtype: float64


In [19]:
print(df["Stock levels"].eq(0).sum())

1


In [21]:

df["Risk Category"] = pd.cut(
    df["Stockout Risk Score"],
    bins=[0, 100, 300, float('inf')],
    labels=["Low", "Medium", "High"]
)

In [23]:
print(df["Risk Category"].value_counts())

Risk Category
Low       44
High      28
Medium    27
Name: count, dtype: int64


In [25]:
print(df.columns.tolist())

['Product type', 'SKU', 'Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Customer demographics', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location', 'Lead time', 'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Inspection results', 'Defect rates', 'Transportation modes', 'Routes', 'Costs', 'Inventory Turnover Proxy', 'Stockout Risk Score', 'Excess Inventory Flag', 'Total Operational Cost', 'Supplier Quality Score', 'Risk Category']


In [32]:
df.to_csv("cleaned_supply_chain.csv", index=False)

print("File saved successfully!")

File saved successfully!


In [27]:
# =====================================
# Automated Report Generation
# =====================================

In [29]:
high_risk = df[df["Risk Category"] == "High"]

high_risk_report = high_risk[
    [
        "SKU",
        "Product type",
        "Stockout Risk Score",
        "Risk Category"
    ]
]

high_risk_report.to_csv(
    "high_risk_skus.csv",
    index=False
)

print("High Risk SKU Report Generated")

High Risk SKU Report Generated


In [31]:
excess_inventory = df[
    df["Excess Inventory Flag"] == "Yes"
]

excess_inventory_report = excess_inventory[
    [
        "SKU",
        "Product type",
        "Stock levels",
        "Number of products sold",
        "Excess Inventory Flag"
    ]
]

excess_inventory_report.to_csv(
    "excess_inventory_products.csv",
    index=False
)

print("Excess Inventory Report Generated")

Excess Inventory Report Generated


In [33]:
supplier_report = (
    df.groupby("Supplier name")
      .agg({
          "Lead times": "mean",
          "Defect rates": "mean",
          "Supplier Quality Score": "mean"
      })
      .reset_index()
)

supplier_report.to_csv(
    "supplier_risk_report.csv",
    index=False
)

print("Supplier Risk Report Generated")

Supplier Risk Report Generated


In [41]:
%run automated_inventory_pipeline.py

Supply Chain Automation Pipeline Started
Dataset Loaded Successfully
Rows: 100
Columns: 31
High Risk Report Generated
Excess Inventory Report Generated
Supplier Risk Report Generated
AUTOMATION PIPELINE COMPLETED
